# QSHIELD interactive audit

Scan quantum circuits for supply-chain tampering, export findings, and sanity-check them against a trusted reference — all inline, no cloud account required.

Requires `pip install qshield` (or `pip install -e .` from the repo). All fixture circuits below ship with the repository.

In [1]:
import json
from pathlib import Path

import qshield
from qshield import Scanner

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEAN = REPO / 'tests' / 'fixtures' / 'clean_circuits' / 'bell_state.qasm'
MALICIOUS = REPO / 'tests' / 'fixtures' / 'malicious_circuits' / 'ancilla_leakage.qasm'
print('qshield', qshield.__version__)
print('fixtures present:', CLEAN.exists(), MALICIOUS.exists())

qshield 0.1.0
fixtures present: True True


In [2]:
clean_results = qshield.scan(str(CLEAN), name='bell_state.qasm')
print('clean circuit findings:', clean_results.summary.to_dict())
print('is_clean:', clean_results.is_clean)

clean circuit findings: {'total': 0, 'critical': 0, 'high': 0, 'medium': 0, 'low': 0, 'info': 0, 'circuits_scanned': 1, 'analyses_run': ['static', 'supply_chain'], 'generated_at': '2026-11-12T17:40:31+0000'}
is_clean: True


In [3]:
sus_results = qshield.scan(str(MALICIOUS), name='ancilla_leakage.qasm')
for finding in sus_results.findings:
    print(f"[{finding.severity.upper():8}] {finding.threat_type} {finding.title}")
print()
print('summary:', sus_results.summary.to_dict())
print('exceeds fail-on=high:', sus_results.exceeds_fail_on('high'))

[HIGH    ] T2.1 Ancilla carries information about the computation

summary: {'total': 1, 'critical': 0, 'high': 1, 'medium': 0, 'low': 0, 'info': 0, 'circuits_scanned': 1, 'analyses_run': ['static', 'supply_chain'], 'generated_at': '2026-11-12T17:40:31+0000'}
exceeds fail-on=high: True


## Exporting for CI/dashboards

`to_sarif()` produces SARIF 2.1.0 (GitHub code scanning / compatible dashboards); `to_json()` gives a plain dict for your own tooling.

In [4]:
sarif_doc = json.loads(sus_results.to_sarif())
print('sarif version:', sarif_doc['version'], '| runs:', len(sarif_doc['runs']), '| results:', len(sarif_doc['runs'][0]['results']))
first = json.loads(sus_results.to_json())['findings'][0]
print('first finding keys:', sorted(first))

sarif version: 2.1.0 | runs: 1 | results: 1
first finding keys: ['analysis_type', 'circuit_id', 'confidence', 'description', 'evidence', 'false_positive_probability', 'finding_id', 'location', 'references', 'remediation', 'severity', 'severity_rank', 'signature_id', 'threat_type', 'title']


## Runtime check on an intercepted circuit (M4.1)

`qshield runtime check` (CLI) verifies a file's commitment hash and static threshold offline. The library equivalent:

In [5]:
from qshield.core.parsers import parse_qasm
from qshield.runtime.integrity import canonical_circuit_hash, HASH_HEADER
from qshield.runtime.validation import validate_counts

ir = parse_qasm(CLEAN.read_text())
print('commitment hash (send as %s):' % HASH_HEADER)
print(canonical_circuit_hash(ir))

honest_counts = {'00': 500, '11': 500}
rigged_counts = {'00': 1000}
print('honest-looking verdict:', validate_counts(ir, honest_counts).status)
print('rigged verdict      :', validate_counts(ir, rigged_counts).status)

commitment hash (send as X-QShield-Expected-Hash):
e287378a28507f58b2fe0075053a43b4749b86dceaf2ccd93cbdef57d2331ff6
honest-looking verdict: match
rigged verdict      : mismatch


## Reading the verdict

- `match`: counts are statistically consistent with the exact (ideal) Born distribution.
- `mismatch`: counts deviate — on ideal/noiseless backends this is a red flag; on noisy hardware it is *drift*, not an attack (see `docs/case_studies/quantum_cloud_audit.md`).
- `unavailable`: QSHIELD refuses to guess (e.g., >12 qubits, classically-conditional gates, resets).

A clean bill of health from QSHIELD means *no evidence of known tampering* — it is not a proof of device honesty.